<a href="https://colab.research.google.com/github/DCMabel/Grupo05_CIIN1021P_EF_REPO/blob/main/Copia_de_DataSalud_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install pyspark==4.0.0 findspark -q

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time
import pandas as pd

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("DIRESA_LaLibertad_BigData_Analisis") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Sesión de Spark creada exitosamente.")
print(f"Versión de Spark: {spark.version}")
print(f"Núcleos disponibles: {spark.sparkContext.defaultParallelism}")

Sesión de Spark creada exitosamente.
Versión de Spark: 4.0.0
Núcleos disponibles: 2


In [ ]:
contenido_csv = """ID;DNI;Edad;Provincia;Diagnostico_Nutricional;Fecha_Atencion
1;72345678;28;Ascope;Sobrepeso;15/02/2024
2;73456789;45;Chepén;Normal;10/03/2024
3;74567890;52;Pacasmayo;Obesidad Tipo I;5/04/2024
4;75678901;37;Trujillo;Obesidad Mórbida;22/01/2024
5;76789012;19;Virú;Normal;12/05/2024"""

with open("atenciones_nutricionales.csv", "w", encoding="utf-8") as archivo:
    archivo.write(contenido_csv)

In [ ]:
# Corrección para la Celda 2:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

esquema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("DNI", StringType(), True),
    StructField("Edad", IntegerType(), True),
    StructField("Provincia", StringType(), True),
    StructField("Diagnostico_Nutricional", StringType(), True),
    StructField("Fecha_Atencion", StringType(), True)
])

df_atenciones = spark.read.csv(
    "atenciones_nutricionales.csv",
    header=False,
    schema=esquema,
    sep=";"
)

In [ ]:
# Celda 3: Análisis con DataFrame API
print("=" * 70)
print("ANÁLISIS 1: DataFrame API - Distribución por Provincia y Diagnóstico")
print("=" * 70)

df_agrupado = df_atenciones \
    .groupBy("Provincia", "Diagnostico_Nutricional") \
    .agg(
        count("*").alias("Total_Atenciones"),
        round(avg("Edad"), 2).alias("Edad_Promedio"),
        min("Fecha_Atencion").alias("Primera_Atencion"),
        max("Fecha_Atencion").alias("Ultima_Atencion")
    ) \
    .orderBy("Provincia", desc("Total_Atenciones"))

print("\nDistribución de atenciones por provincia y diagnóstico:")
df_agrupado.show(20, truncate=False)

df_agrupado.createOrReplaceTempView("vista_agrupada")

ANÁLISIS 1: DataFrame API - Distribución por Provincia y Diagnóstico

Distribución de atenciones por provincia y diagnóstico:
+---------+-----------------------+----------------+-------------+----------------+---------------+
|Provincia|Diagnostico_Nutricional|Total_Atenciones|Edad_Promedio|Primera_Atencion|Ultima_Atencion|
+---------+-----------------------+----------------+-------------+----------------+---------------+
|Ascope   |Sobrepeso              |1               |28.0         |15/02/2024      |15/02/2024     |
|Chepén   |Normal                 |1               |45.0         |10/03/2024      |10/03/2024     |
|Pacasmayo|Obesidad Tipo I        |1               |52.0         |5/04/2024       |5/04/2024      |
|Provincia|Diagnostico_Nutricional|1               |NULL         |Fecha_Atencion  |Fecha_Atencion |
|Trujillo |Obesidad Mórbida       |1               |37.0         |22/01/2024      |22/01/2024     |
|Virú     |Normal                 |1               |19.0         |12/05/20

In [ ]:
# Celda 4: Análisis con SparkSQL
print("=" * 70)
print("ANÁLISIS 2: SparkSQL - Clasificación de Riesgo y Morbilidad")
print("=" * 70)

# Registrar DataFrame como vista temporal
df_atenciones.createOrReplaceTempView("atenciones_nutricionales")

consulta_sparksql = """
SELECT
    Provincia,
    Diagnostico_Nutricional,
    COUNT(*) AS Total_Atenciones,
    ROUND(AVG(Edad), 2) AS Edad_Promedio,
    CASE
        WHEN Diagnostico_Nutricional = 'Normal' THEN 'Bajo Riesgo'
        WHEN Diagnostico_Nutricional = 'Sobrepeso' THEN 'Riesgo Moderado'
        WHEN Diagnostico_Nutricional = 'Obesidad Tipo I' THEN 'Riesgo Alto'
        WHEN Diagnostico_Nutricional = 'Obesidad Tipo II' THEN 'Riesgo Muy Alto'
        WHEN Diagnostico_Nutricional = 'Obesidad Mórbida' THEN 'Riesgo Crítico'
        ELSE 'No Clasificado'
    END AS Nivel_Riesgo
FROM atenciones_nutricionales
GROUP BY Provincia, Diagnostico_Nutricional
ORDER BY Total_Atenciones DESC
"""

df_resultado_sparksql = spark.sql(consulta_sparksql)
print("\nResultado de consulta SparkSQL con clasificación de riesgo:")
df_resultado_sparksql.show(20, truncate=False)

ANÁLISIS 2: SparkSQL - Clasificación de Riesgo y Morbilidad

Resultado de consulta SparkSQL con clasificación de riesgo:
+---------+-----------------------+----------------+-------------+---------------+
|Provincia|Diagnostico_Nutricional|Total_Atenciones|Edad_Promedio|Nivel_Riesgo   |
+---------+-----------------------+----------------+-------------+---------------+
|Virú     |Normal                 |1               |19.0         |Bajo Riesgo    |
|Pacasmayo|Obesidad Tipo I        |1               |52.0         |Riesgo Alto    |
|Chepén   |Normal                 |1               |45.0         |Bajo Riesgo    |
|Trujillo |Obesidad Mórbida       |1               |37.0         |Riesgo Crítico |
|Provincia|Diagnostico_Nutricional|1               |NULL         |No Clasificado |
|Ascope   |Sobrepeso              |1               |28.0         |Riesgo Moderado|
+---------+-----------------------+----------------+-------------+---------------+



In [ ]:
# Celda 5: Estadística descriptiva completa
print("=" * 70)
print("ESTADÍSTICA DESCRIPTIVA - Dataset DIRESA La Libertad")
print("=" * 70)

# 1. Estadísticas numéricas de Edad
print("\nESTADÍSTICAS DESCRIPTIVAS DE EDAD:")
df_atenciones.select("Edad").describe().show()

# 2. Distribución por nivel de riesgo
print("\nDISTRIBUCIÓN POR NIVEL DE RIESGO:")
df_nivel_riesgo = df_resultado_sparksql.groupBy("Nivel_Riesgo") \
    .agg(
        count("*").alias("Cantidad_Diagnosticos"),
        sum("Total_Atenciones").alias("Total_Pacientes")
    ) \
    .orderBy(desc("Total_Pacientes"))

df_nivel_riesgo.show(truncate=False)

# 3. Provincia con mayor carga de morbilidad
print("\n3PROVINCIA CON MAYOR CARGA DE MORBILIDAD (Riesgo Alto+):")
df_morbilidad = spark.sql("""
    SELECT
        Provincia,
        COUNT(*) AS Total_Casos_Alto_Riesgo,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM atenciones_nutricionales), 2) AS Porcentaje
    FROM (
        SELECT
            Provincia,
            CASE
                WHEN Diagnostico_Nutricional IN ('Obesidad Tipo I', 'Obesidad Tipo II', 'Obesidad Mórbida')
                THEN 'Alto'
                ELSE 'Bajo'
            END AS Categoria_Riesgo
        FROM atenciones_nutricionales
    )
    WHERE Categoria_Riesgo = 'Alto'
    GROUP BY Provincia
    ORDER BY Total_Casos_Alto_Riesgo DESC
""")
df_morbilidad.show(truncate=False)

# 4. Resumen ejecutivo
print("\nRESUMEN EJECUTIVO:")
total_registros = df_atenciones.count()
total_provincias = df_atenciones.select("Provincia").distinct().count()
edad_promedio = df_atenciones.agg(round(avg("Edad"), 2)).collect()[0][0]

print(f"• Total de atenciones analizadas: {total_registros}")
print(f"• Provincias cubiertas: {total_provincias}")
print(f"• Edad promedio de pacientes: {edad_promedio} años")

ESTADÍSTICA DESCRIPTIVA - Dataset DIRESA La Libertad

ESTADÍSTICAS DESCRIPTIVAS DE EDAD:
+-------+------------------+
|summary|              Edad|
+-------+------------------+
|  count|                 5|
|   mean|              36.2|
| stddev|13.141537200799608|
|    min|                19|
|    max|                52|
+-------+------------------+


DISTRIBUCIÓN POR NIVEL DE RIESGO:
+---------------+---------------------+---------------+
|Nivel_Riesgo   |Cantidad_Diagnosticos|Total_Pacientes|
+---------------+---------------------+---------------+
|Bajo Riesgo    |2                    |2              |
|Riesgo Alto    |1                    |1              |
|Riesgo Moderado|1                    |1              |
|Riesgo Crítico |1                    |1              |
|No Clasificado |1                    |1              |
+---------------+---------------------+---------------+


3PROVINCIA CON MAYOR CARGA DE MORBILIDAD (Riesgo Alto+):
+---------+-----------------------+----------+
|Pro

In [ ]:
# Celda 6: Benchmark de PySpark
print("=" * 70)
print("BENCHMARK: Comparación de Tiempos de Ejecución")
print("=" * 70)

# Consulta de agregación compleja
consulta_benchmark = """
SELECT
    Provincia,
    COUNT(*) AS Total,
    ROUND(AVG(Edad), 2) AS Edad_Prom,
    COUNT(CASE WHEN Diagnostico_Nutricional IN ('Obesidad Tipo I', 'Obesidad Tipo II', 'Obesidad Mórbida')
         THEN 1 END) AS Casos_Obesidad
FROM atenciones_nutricionales
GROUP BY Provincia
ORDER BY Total DESC
"""

# Medir tiempo de ejecución en Spark
print("\n Ejecutando consulta en PySpark...")
inicio_spark = time.time()
df_benchmark_spark = spark.sql(consulta_benchmark)
df_benchmark_spark.collect()
fin_spark = time.time()

tiempo_spark = fin_spark - inicio_spark
print(f"Tiempo de ejecución en PySpark: {tiempo_spark:.4f} segundos")
print("\nResultado:")
df_benchmark_spark.show(truncate=False)

BENCHMARK: Comparación de Tiempos de Ejecución

 Ejecutando consulta en PySpark...
Tiempo de ejecución en PySpark: 1.4998 segundos

Resultado:
+---------+-----+---------+--------------+
|Provincia|Total|Edad_Prom|Casos_Obesidad|
+---------+-----+---------+--------------+
|Virú     |1    |19.0     |0             |
|Provincia|1    |NULL     |0             |
|Ascope   |1    |28.0     |0             |
|Chepén   |1    |45.0     |0             |
|Trujillo |1    |37.0     |1             |
|Pacasmayo|1    |52.0     |1             |
+---------+-----+---------+--------------+



In [ ]:
# Celda 8: Tabla comparativa
print("=" * 70)
print("TABLA COMPARATIVA: SQL Server vs PySpark")
print("=" * 70)

# Reemplaza estos valores con los de tu ejecución real
tiempo_sql_server = 0.042

datos_comparacion = {
    'Métrica': [
        'Tiempo de ejecución (segundos)',
        'Tipo de procesamiento',
        'Escalabilidad horizontal',
        'Modelo de datos',
        'Adecuación para 900 registros',
        'Adecuación para 10 millones de registros',
        'Costo de infraestructura',
        'Facilidad de integración con BI'
    ],
    'SQL Server': [
        f'{tiempo_sql_server:.4f}',
        'Procesamiento en memoria del servidor (single-node)',
        'Limitada (Scale-Up vertical: más RAM/CPU)',
        'Relacional rígido (tablas con esquema fijo)',
        'Óptimo - respuesta instantánea',
        '️ Requiere hardware costoso o particionamiento',
        'Licencia comercial ($$$)',
        'Nativa con Power BI'
    ],
    'PySpark': [
        f'{tiempo_spark:.4f}',
        'Procesamiento distribuido en cluster (multi-node)',
        'Nativa (Scale-Out horizontal: más nodos)',
        'Flexible (DataFrames + RDDs + SQL)',
        'Overhead de inicialización de Spark',
        'Óptimo - distribuye carga automáticamente',
        'Open source (gratuito)',
        '️ Requiere conectores adicionales'
    ]
}

df_comparacion = pd.DataFrame(datos_comparacion)
print(df_comparacion.to_string(index=False))


df_comparacion.to_csv("tabla_comparativa_sql_vs_spark.csv", index=False)
print("\nTabla guardada como 'tabla_comparativa_sql_vs_spark.csv'")

TABLA COMPARATIVA: SQL Server vs PySpark
                                 Métrica                                          SQL Server                                           PySpark
          Tiempo de ejecución (segundos)                                              0.0420                                            1.4998
                   Tipo de procesamiento Procesamiento en memoria del servidor (single-node) Procesamiento distribuido en cluster (multi-node)
                Escalabilidad horizontal           Limitada (Scale-Up vertical: más RAM/CPU)          Nativa (Scale-Out horizontal: más nodos)
                         Modelo de datos         Relacional rígido (tablas con esquema fijo)                Flexible (DataFrames + RDDs + SQL)
           Adecuación para 900 registros                      Óptimo - respuesta instantánea               Overhead de inicialización de Spark
Adecuación para 10 millones de registros      ️ Requiere hardware costoso o particionamiento         

In [ ]:
# 3. RDD: Conteo por diagnóstico mediante map y reduceByKey
print("Conteo por diagnóstico usando RDD:")

rdd_diagnosticos = df_atenciones.rdd \
    .map(lambda fila: (fila["Diagnostico_Nutricional"], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()

for diagnostico, total in rdd_diagnosticos:
    print(f"- {diagnostico}: {total}")

Conteo por diagnóstico usando RDD:
- Diagnostico_Nutricional: 1
- Sobrepeso: 1
- Normal: 2
- Obesidad Tipo I: 1
- Obesidad Mórbida: 1
